In [ ]:
import os
import requests
import random
import time
from PIL import Image
from io import BytesIO

# --- KONFIGURACJA ---
SAVE_DIR = "lunar_surface_dataset"
TARGET_COUNT = 100
IMAGE_SIZE = 512 # Pobieramy kwadraty 512x512 pikseli

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# Endpoint serwera WMS USGS
WMS_URL = "https://planetarymaps.usgs.gov/cgi-bin/mapserv"

# TUTAJ BYŁ BŁĄD: Musi być "earth", serwery USGS tak katalogują mapę Księżyca
WMS_MAP = "/maps/earth/moon_simp_cyl.map" 
LAYER = "LROC_WAC" # Warstwa z Lunar Reconnaissance Orbiter

In [ ]:
print(f"Rozpoczynam pobieranie {TARGET_COUNT} losowych wycinków satelitarnych Księżyca...")

for i in range(TARGET_COUNT):
    # Losowanie współrzędnych
    lon_min = random.uniform(-170, 160)
    lat_min = random.uniform(-60, 60)
    
    delta = 0.5 # Obszar w stopniach
    lon_max = lon_min + delta
    lat_max = lat_min + delta
    
    # Kompletne parametry zapytania WMS 1.1.1
    params = {
        "map": WMS_MAP,
        "request": "GetMap",
        "service": "WMS",
        "version": "1.1.1",
        "layers": LAYER,
        "styles": "",  # Wymagany przez protokół WMS
        "srs": "EPSG:4326",
        "bbox": f"{lon_min},{lat_min},{lon_max},{lat_max}",
        "width": IMAGE_SIZE,
        "height": IMAGE_SIZE,
        "format": "image/jpeg"
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    try:
        response = requests.get(WMS_URL, params=params, headers=headers, timeout=15)
        response.raise_for_status()
        
        # Sprawdzenie, czy serwer zwrócił obraz
        if 'image' in response.headers.get('Content-Type', ''):
            img = Image.open(BytesIO(response.content))
            
            # Wymuszenie odcieni szarości
            img = img.convert("L") 
            
            file_path = os.path.join(SAVE_DIR, f"lroc_tile_{i+1:03d}.jpg")
            img.save(file_path, "JPEG")
            
            print(f"[{i+1}/{TARGET_COUNT}] Zapisano: (Lon: {lon_min:.2f}, Lat: {lat_min:.2f})")
        else:
            error_snippet = response.text[:250].replace('\n', ' ')
            print(f"[{i+1}/{TARGET_COUNT}] Błąd WMS! Odpowiedź serwera: {error_snippet}")
            
    except Exception as e:
        print(f"[{i+1}/{TARGET_COUNT}] Błąd sieci: {e}")
        
    time.sleep(1.0)

print(f"\nGotowe! Dataset został zapisany w folderze '{SAVE_DIR}'.")

Rozpoczynam pobieranie 100 losowych wycinków satelitarnych Księżyca...
[1/100] Zapisano: (Lon: 63.46, Lat: -55.50)
[2/100] Zapisano: (Lon: -147.93, Lat: 53.66)
[3/100] Zapisano: (Lon: -164.73, Lat: 11.74)
[4/100] Zapisano: (Lon: -26.22, Lat: 15.92)
[5/100] Zapisano: (Lon: 95.02, Lat: 20.03)
[6/100] Zapisano: (Lon: -164.15, Lat: 51.72)
[7/100] Zapisano: (Lon: 80.36, Lat: -3.77)
[8/100] Zapisano: (Lon: 40.31, Lat: 42.11)
[9/100] Zapisano: (Lon: 127.48, Lat: -41.97)
[10/100] Zapisano: (Lon: -28.30, Lat: 13.61)
[11/100] Zapisano: (Lon: -167.86, Lat: 41.55)
[12/100] Zapisano: (Lon: 66.30, Lat: -3.59)
[13/100] Zapisano: (Lon: -105.22, Lat: -2.08)
[14/100] Zapisano: (Lon: -113.18, Lat: 58.89)
[15/100] Zapisano: (Lon: -131.21, Lat: 37.51)
[16/100] Zapisano: (Lon: -76.86, Lat: -14.06)
[17/100] Zapisano: (Lon: 122.66, Lat: 3.69)
[18/100] Zapisano: (Lon: 42.25, Lat: 37.47)
[19/100] Zapisano: (Lon: 99.51, Lat: -29.78)
[20/100] Zapisano: (Lon: 108.46, Lat: 55.99)
[21/100] Zapisano: (Lon: 1.07, Lat: